**Fig. 6 and 7, step 2 (notebook 02): build the niche embedding over each stack's cell contact graph (the figures use hop3_pure at resolution 1.5)**

In [ ]:
# Working directory: data/fig6_7/ under the repository root, where the Fig. 6 and 7 deposit folder is downloaded
# (all_stacks_features.csv, workspace/ = the plugin's Step 4 workspace with the 14 stacks (13 analysed) and their meshes,
# run/ = what these notebooks write and read back, figures/ = the saved panels).
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig6_7")
print("working directory:", os.getcwd())

# 02 -- Spatial Niche Embedding

Neighborhood PC aggregation using adjacency graphs. Each cell's niche embedding is a
weighted average of its own PCs and its neighbors' PCs (diffusion on the contact graph).

**Inputs:** `morphospace_results.pkl`, adjacency CSVs (auto-discovered from state.json)
**Outputs:** `niche_results.pkl`

Multiple niche configurations are compared: varying hop count, self-weight, and
pure-neighbor vs self-inclusive modes.

In [ ]:
import os, json, glob, pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import sparse
from sklearn.neighbors import NearestNeighbors
from umap import UMAP
import igraph as ig
import leidenalg
warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
from itertools import product

# -- PATHS --
OUTPUT_DIR    = r'run'
WORKSPACE_DIR = r'workspace'

# -- NICHE PARAMETER AXES --
SELF_WEIGHTS = [0.0, 0.3, 0.5, 0.7]
N_ITERS      = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

def _make_name(w, k):
    return f'hop{k}_pure' if w == 0.0 else f'hop{k}_w{w}'

NICHE_CONFIGS = [
    {'self_weight': w, 'n_iter': k, 'name': _make_name(w, k)}
    for w, k in product(SELF_WEIGHTS, N_ITERS)
]

# -- OTHER PARAMS --
WEIGHT_BY_CONTACT = True
MIN_CONTACT       = 100  # default
NICHE_LEIDEN_RES  = [0.3, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
RANDOM_SEED       = 42

print(f'Niche configs: {len(NICHE_CONFIGS)}')          # 40 configs
print(f'Niche Leiden resolutions: {NICHE_LEIDEN_RES}') # 7
print(f'Total clusterings: {len(NICHE_CONFIGS) * len(NICHE_LEIDEN_RES)}')  # 280 clusterings (40 configs x 7 resolutions)

## Load Morphospace Results

In [ ]:
pkl_path = os.path.join(OUTPUT_DIR, 'morphospace_results.pkl')
with open(pkl_path, 'rb') as f:
    results = pickle.load(f)

df = results['df']
pcs = results['pcs']
leiden_results = results['leiden_results']
config = results['config']
CLUSTER_CMAP = results['CLUSTER_CMAP']
META_MAP = results['META_MAP']
BG = results['BG']
AX_BG = results['AX_BG']
TICK = results['TICK']

# Rebuild color helpers
_BASE_PALETTE = [
    '#E74C3C', '#3498DB', '#2ECC71', '#E67E22', '#9B59B6',
    '#F39C12', '#1ABC9C', '#E91E63', '#FF6B6B', '#45B7D1',
    '#A29BFE', '#FD79A8', '#FDCB6E', '#6C5CE7', '#00B894',
    '#D63031', '#0984E3', '#00CEC9', '#E17055', '#74B9FF',
]

def _generate_colors(n):
    if n <= len(_BASE_PALETTE):
        return _BASE_PALETTE[:n]
    import colorsys
    colors = list(_BASE_PALETTE)
    for i in range(n - len(_BASE_PALETTE)):
        hue = (i * 0.618033988749895) % 1.0
        r, g, b = colorsys.hsv_to_rgb(hue, 0.7 + (i % 3) * 0.1, 0.85 + (i % 2) * 0.1)
        colors.append('#' + '%02x%02x%02x' % (int(r*255), int(g*255), int(b*255)))
    return colors

def build_cluster_cmap(clusters):
    sorted_c = sorted(clusters)
    colors = _generate_colors(len(sorted_c))
    return {c: colors[i] for i, c in enumerate(sorted_c)}

print('Loaded: ' + str(len(df)) + ' cells, ' + str(pcs.shape[1]) + ' PCs')
print('Leiden resolutions: ' + str(list(leiden_results.keys())))

## Build Niche Embeddings (multiple configs)

For each niche configuration, diffuse PCs on the per-stack adjacency graph.

In [ ]:
# Load state.json for adjacency path discovery
state_path = os.path.join(WORKSPACE_DIR, 'stack_segmentation_state.json')
with open(state_path) as _f:
    _state = json.load(_f)

n_pcs = pcs.shape[1]
niche_results = {}

for nc in NICHE_CONFIGS:
    nc_name = nc['name']
    self_weight = nc['self_weight']
    n_iter = nc['n_iter']
    print('\n' + '=' * 60)
    print('Niche config: ' + nc_name + ' (self_weight=' + str(self_weight) + ', n_iter=' + str(n_iter) + ')')
    print('=' * 60)

    niche_matrix = np.full_like(pcs, np.nan, dtype=np.float64)
    stats = []

    for stack_name in sorted(df['stack_name'].unique()):
        mask = (df['stack_name'] == stack_name).values
        global_idx = np.where(mask)[0]
        n_cells = len(global_idx)
        if n_cells == 0:
            continue

        sub = df.iloc[global_idx]
        label_to_local = {int(lbl): i for i, lbl in enumerate(sub['label'].values)}

        record = _state['stacks'].get(stack_name, {})
        features_csv = record.get('qc_features_csv', '')
        if not features_csv:
            continue
        if not os.path.isabs(features_csv):          # recorded relative to the workspace root
            features_csv = os.path.join(WORKSPACE_DIR, features_csv)
        qc_dir = os.path.dirname(features_csv)
        adj_path = os.path.join(qc_dir, stack_name + '_adjacency.csv')
        if not os.path.exists(adj_path):
            continue

        adj = pd.read_csv(adj_path)
        n_total = len(adj)
        adj = adj[adj['contact_voxels'] >= MIN_CONTACT]
        a_ok = adj['label_a'].astype(int).isin(label_to_local)
        b_ok = adj['label_b'].astype(int).isin(label_to_local)
        adj = adj[a_ok & b_ok]

        rows_sp = adj['label_a'].astype(int).map(label_to_local).values
        cols_sp = adj['label_b'].astype(int).map(label_to_local).values
        if WEIGHT_BY_CONTACT:
            w = adj['contact_voxels'].astype(float).values
        else:
            w = np.ones(len(adj), dtype=float)

        W = sparse.coo_matrix(
            (np.concatenate([w, w]),
             (np.concatenate([rows_sp, cols_sp]), np.concatenate([cols_sp, rows_sp]))),
            shape=(n_cells, n_cells),
        ).tocsr()

        row_sums = np.asarray(W.sum(axis=1)).flatten()
        degree_zero = (row_sums == 0)
        row_sums[degree_zero] = 1.0
        inv_sums = sparse.diags(1.0 / row_sums)
        W_norm = inv_sums @ W

        X_self = pcs[global_idx].copy()
        X_niche = X_self.copy()
        for _ in range(n_iter):
            X_neigh = W_norm @ X_niche
            X_niche = self_weight * X_self + (1.0 - self_weight) * X_neigh
            if degree_zero.any():
                X_niche[degree_zero] = X_self[degree_zero]

        niche_matrix[global_idx] = X_niche
        stats.append({
            'stack': stack_name.replace('_Smooth05_Resampled.tif', ''),
            'cells': n_cells,
            'edges_total': n_total,
            'edges_kept': len(adj),
        })

    n_valid = int((~np.isnan(niche_matrix[:, 0])).sum())
    print('Valid cells: ' + str(n_valid) + '/' + str(len(df)))
    if not any(s in _state['stacks'] for s in df['stack_name'].unique()):
        raise RuntimeError('no stack of the feature table is a key of stack_segmentation_state.json: '
                           'the workspace and the feature table must use the same stack names')
    n_stacks = df['stack_name'].nunique()
    if not stats:
        raise RuntimeError('no stack has an adjacency table (a qc_features_csv in stack_segmentation_state.json '
                           'with <stack>_adjacency.csv next to it): no cell gets niche coordinates')
    if len(stats) < n_stacks and nc is NICHE_CONFIGS[0]:        # the coverage is the same for every config
        print('WARNING: ' + str(n_stacks - len(stats)) + ' of ' + str(n_stacks) + ' stacks have no state record '
              'or no adjacency table and get no niche coordinates')
    niche_results[nc_name] = {
        'niche_pcs': niche_matrix,
        'config': nc,
        'stats': stats,
        'n_valid': n_valid,
    }

print('\nAll niche configs computed.')

## Niche UMAP + Leiden (per config)

In [ ]:
for nc_name, nr in niche_results.items():
    niche_matrix = nr['niche_pcs']
    valid_mask = ~np.isnan(niche_matrix[:, 0])
    X_valid = niche_matrix[valid_mask]
    print('\n--- ' + nc_name + ' (' + str(len(X_valid)) + ' cells) ---')

    # UMAP
    niche_umap = UMAP(
        n_components=2, n_neighbors=15, min_dist=0.1,
        metric='euclidean', random_state=RANDOM_SEED,
    ).fit_transform(X_valid)
    nr['umap'] = niche_umap
    nr['valid_mask'] = valid_mask
    print('  UMAP done')

    # KNN graph
    nn = NearestNeighbors(n_neighbors=16, metric='euclidean').fit(X_valid)
    dists, idxs = nn.kneighbors(X_valid)
    edges_, weights_ = [], []
    for i in range(len(X_valid)):
        for j, d in zip(idxs[i, 1:], dists[i, 1:]):
            edges_.append((i, int(j)))
            weights_.append(float(np.exp(-d)))
    G_niche = ig.Graph(n=len(X_valid), edges=edges_, directed=False)
    G_niche.es['weight'] = weights_
    G_niche = G_niche.simplify(combine_edges='mean')

    # Multi-res Leiden
    nr['clusters'] = {}
    for res in NICHE_LEIDEN_RES:
        part = leidenalg.find_partition(
            G_niche, leidenalg.RBConfigurationVertexPartition,
            weights='weight', resolution_parameter=res, seed=RANDOM_SEED,
        )
        nr['clusters'][res] = {
            'membership': np.array(part.membership),
            'n_clusters': len(set(part.membership)),
        }
        print('  Leiden res=' + str(res) + ': ' + str(len(set(part.membership))) + ' clusters')

print('\nAll niche embeddings + clustering done.')

## Comparison Grid

In [ ]:
n_configs = len(niche_results)
n_res = len(NICHE_LEIDEN_RES)

fig, axes = plt.subplots(n_configs, n_res, figsize=(6 * n_res, 5 * n_configs), facecolor=BG)
if n_configs == 1:
    axes = axes.reshape(1, -1)
if n_res == 1:
    axes = axes.reshape(-1, 1)

for row_i, (nc_name, nr) in enumerate(niche_results.items()):
    umap_coords = nr['umap']
    for col_i, res in enumerate(NICHE_LEIDEN_RES):
        ax = axes[row_i, col_i]
        ax.set_facecolor(AX_BG)
        ax.tick_params(colors=TICK, labelsize=6)

        membership = nr['clusters'][res]['membership']
        n_c = nr['clusters'][res]['n_clusters']
        cmap = build_cluster_cmap(set(membership))
        colors = [cmap[c] for c in membership]

        ax.scatter(umap_coords[:, 0], umap_coords[:, 1],
            c=colors, s=2, alpha=0.5, linewidths=0, rasterized=True)
        title = nc_name + ' | res=' + str(res) + ' (' + str(n_c) + 'c)'
        ax.set_title(title, fontsize=8, color='#DDDDDD', pad=3)

fig.suptitle('Niche UMAP x Leiden resolution', fontsize=12, color='#CCCCCC', y=1.01)
plt.tight_layout(); plt.show()

## Sample Niche Representatives

In [ ]:
mesh_lookup = results.get('mesh_lookup', {})

for nc_name, nr in niche_results.items():
    valid_mask = nr['valid_mask']
    valid_idx = np.where(valid_mask)[0]
    niche_pcs_valid = nr['niche_pcs'][valid_mask]
    df_valid = df.iloc[valid_idx].copy().reset_index(drop=True)

    for res in NICHE_LEIDEN_RES:
        membership = nr['clusters'][res]['membership']
        df_valid['niche_cluster'] = membership

        rep_rows = []
        for c in sorted(set(membership)):
            mask_c = (membership == c)
            idx_c = np.where(mask_c)[0]
            pcs_c = niche_pcs_valid[mask_c]
            median = np.median(pcs_c, axis=0)
            dists_c = np.linalg.norm(pcs_c - median, axis=1)
            n_pick = min(10, len(idx_c))
            closest = idx_c[np.argsort(dists_c)[:n_pick]]
            for rank, ri in enumerate(closest):
                row = df_valid.iloc[ri]
                sname = row['stack_name']
                lbl = int(row['label'])
                mpath = mesh_lookup.get((sname, lbl), None)
                rep_rows.append({
                    'cluster': int(c),
                    'role': 'rep_' + str(rank),
                    'rank': rank,
                    'label': lbl,
                    'stack_name': sname,
                    'mesh_path': mpath if mpath else '',
                    'mesh_exists': mpath is not None and os.path.exists(str(mpath)),
                })

        out_dir = os.path.join(OUTPUT_DIR, 'niche_' + nc_name, 'leiden_res_' + str(res))
        os.makedirs(out_dir, exist_ok=True)
        rep_df = pd.DataFrame(rep_rows)
        rep_df.to_csv(os.path.join(out_dir, 'representative_cells.csv'), index=False)
        n_found = rep_df['mesh_exists'].sum()
        print(nc_name + ' res=' + str(res) + ': ' + str(len(rep_df)) + ' reps ('
              + str(n_found) + ' with meshes)')

print('\nAll niche representatives saved.')

## Save Niche Results

In [ ]:
niche_pkl_path = os.path.join(OUTPUT_DIR, 'niche_results.pkl')
with open(niche_pkl_path, 'wb') as f:
    pickle.dump({
        'niche_results': niche_results,
        'NICHE_CONFIGS': NICHE_CONFIGS,
        'NICHE_LEIDEN_RES': NICHE_LEIDEN_RES,
        'WEIGHT_BY_CONTACT': WEIGHT_BY_CONTACT,
        'MIN_CONTACT': MIN_CONTACT,
    }, f)
print('Saved: ' + niche_pkl_path)